# Сценарии электрических свойств лёгкого

**Статус:** контракт расчёта; исполняемая реализация и принятые
численные сценарии отсутствуют.

Этап переводит модельно-условную долю воздуха из 20.03 в набор
частотно-зависимых сценариев эффективных электрических свойств
лёгкого. Результат не является прямым КТ-измерением
$\rho_2$ и не заменяет оценку из электрического эксперимента.


## Происхождение и отклонённые выводы

Исторический источник:
10_КТ_оценка_HU.ipynb в коммите
1ac7813654b59861e359cffe3ad40941baeab2d9.
В нём были объединены DICOM-загрузка, пороговая сегментация,
выбор нижней трети правого лёгкого, формула HU → доля воздуха,
модели Максвелла–Гарнетта и Арчи и сравнение с обратной задачей.
Указатель на смешанную реализацию сохранён в
archive/legacy/20.90_Скетч_пороговой_КТ_сегментации.ipynb.

Исторические значения около 17–19 Ом·м и утверждение о
«независимом подтверждении двуслойной модели без подгонки» не
принимаются. Они зависят от свободно выбранного
$\rho_{\mathrm{matrix}}=2.5$ Ом·м, неподтверждённого ROI,
неизвестной частоты, выбранной модели смешения и результата
другой обратной задачи. Совпадение при таких условиях является
предварительной проверкой порядка величины, а не валидацией.


## Что именно моделируется

Базовая частотно-зависимая величина —
комплексная адмиттивность

$$
\gamma^*(\omega)=\sigma(\omega)+j\omega\varepsilon(\omega),
$$

а комплексное удельное сопротивление определяется как
$\rho^*(\omega)=1/\gamma^*(\omega)$.

Если прибор и аналитическая модель используют только модуль
импеданса, переход к $|\rho^*|$, $\operatorname{Re}\rho^*$ или
вещественному скалярному приближению должен быть выбран и
обоснован до сравнения с данными. Скалярный результат следует
называть rho2_scalar_proxy, а не истинным удельным
сопротивлением лёгкого.

$\rho_{\mathrm{matrix}}$ обозначает эффективное свойство всей
неаэрированной компоненты ROI: паренхиматозной ткани, крови,
жидкости и стенок воздухоносных структур. Это не сопротивление
чистой крови и не табличное свойство одной ткани.


## Сценарий Максвелла–Гарнетта

Для сферических включений с адмиттивностью
$\gamma^*_{\mathrm{air}}$ в непрерывной матрице
$\gamma^*_{\mathrm{matrix}}$:

$$
\gamma^*_{\mathrm{eff}}=
\gamma^*_{\mathrm{matrix}}
\frac{
  \gamma^*_{\mathrm{air}}+2\gamma^*_{\mathrm{matrix}}
  +2f(\gamma^*_{\mathrm{air}}-\gamma^*_{\mathrm{matrix}})
}{
  \gamma^*_{\mathrm{air}}+2\gamma^*_{\mathrm{matrix}}
  -f(\gamma^*_{\mathrm{air}}-\gamma^*_{\mathrm{matrix}})
}.
$$

При идеально изолирующем воздухе и вещественной матрице это
даёт историческую формулу

$$
\rho_{\mathrm{eff}}=
\rho_{\mathrm{matrix}}
\frac{2+f}{2(1-f)}
=
\rho_{\mathrm{matrix}}
\frac{1+f/2}{1-f}.
$$

Это сценарий, а не установленная модель лёгкого. Его условия:
однородная изотропная непрерывная матрица, сферические
невзаимодействующие включения, разделение масштабов и отсутствие
влияния геометрии крупных сосудов и дыхательных путей. При высокой
доле воздуха предположение о разреженных невзаимодействующих
включениях особенно сомнительно и требует отдельной проверки.


## Сценарий типа Арчи

Исторический код использовал эмпирическую зависимость

$$
\rho_{\mathrm{eff}}=
a\,\rho_{\mathrm{matrix}}(1-f)^{-m}.
$$

Здесь проводящей долей объявлена $1-f$. Такое применение является
адаптацией пористо-средового закона к лёгкому, а не прямым
следствием закона Арчи. Коэффициент $a$, показатель $m$, диапазон
частот, температура, топология проводящей фазы и способ калибровки
должны иметь независимое происхождение.

Историческое значение $m=1.5$ не считается принятым. До появления
источника или собственной калибровки этот сценарий допускается
только как анализ чувствительности по заранее заданной сетке
$(a,m)$.


## Нелинейность и масштаб усреднения

Обе зависимости быстро растут при $f\to1$. Для скалярного
сценария Максвелла–Гарнетта

$$
\frac{\partial\rho_{\mathrm{eff}}}{\partial f}
=
\frac{3\rho_{\mathrm{matrix}}}{2(1-f)^2},
$$

поэтому малая ошибка доли воздуха в хорошо аэрированном ROI может
заметно изменить итоговое сопротивление.

Из-за нелинейности в общем случае

$$
T\!\left(\overline f\right)\ne
\overline{T(f)}.
$$

Следовательно, нельзя без проверки преобразовать только средний HU
или среднюю долю воздуха. 20.03 должен передать распределение или
гистограмму $f$. Расчёт обязан раздельно показать:

1. сосредоточенную ROI-модель $T(\overline f)$;
2. статистику воксельного преобразования $T(f)$ как показатель
   чувствительности;
3. различие этих оценок.

Ни одна из них автоматически не является физически корректным
гомогенизированным свойством всего ROI.


## Обязательные входы и область переноса

1. Принятый результат 20.03 с хешами 20.01 и 20.02, дыхательным
   состоянием, определением ROI, распределением $f$ и долей
   значений вне $[0,1]$.
2. Частота или спектр частот конкретного прибора и аппаратной
   ревизии.
3. Комплексные свойства неаэрированной матрицы при согласованных
   частоте и температуре либо явно ограниченный сценарный диапазон.
4. Модель смешения, её параметры, источник, область применимости и
   правило перехода к величине, используемой в 30.01.
5. Для сравнения вдоха и выдоха — два согласованных КТ-состояния
   либо независимо проверенная модель переноса между состояниями.

Историческая КТ относится к добровольцам и не синхронна с
электрическими записями. Первичная предполагаемая сверка относится
к эксперименту 2. Перенос в эксперимент 3 требует отдельной
проверки частоты и реокардиомонитора РНЦХ. Один вдоховый том не
определяет вентиляционное или пульсовое $\Delta\rho_2$.


## Предусмотренный расчёт и выход

До появления частоты и свойств матрицы допустимо рассчитывать
только безразмерные множители
$\rho_{\mathrm{eff}}/\rho_{\mathrm{matrix}}$ и их
чувствительность к $f$. Численные значения в Ом·м в этом случае
блокируются.

После выполнения входного контракта для каждого сценария должны
сохраняться:

- модель, версия формулы и её физические допущения;
- $\gamma^*_{\mathrm{matrix}}$, температура, частота и источник;
- параметры $(a,m)$ или свойства включений;
- результат для принятого КТ-состояния и интервал
  чувствительности;
- раздельные вклады неопределённости $f$, матрицы, модели и
  частоты;
- доля входа вне области применимости и предупреждения;
- хеши всех входных манифестов, версия кода и ручной статус
  candidate или accepted.

Канонический файл должен находиться вне Git:
derived_root/ct/electrical_scenarios/<subject_id>.json.
Исторический params/ct.json не является выходом 20.04.
В серии 30–33 передаётся набор сценариев или предварительное
распределение, но не жёстко заданное «КТ-значение rho2». Слепое
межмодальное сопоставление принадлежит только
34.03_КТ_как_референс_разработки.md.


## Статус литературного обоснования

Проверены профильные коллекции и полнотекстовый индекс локального
Zotero Heart_Lung.

- Zakharchenko, Tikhomirov, *Decomposition of Surface Thoracic
  Impedance to Partial Impedances of Internal Tissues*
  (Zotero QF5XKU75) по описанию FEM-расчёта показывают
  нелинейную монотонную связь поверхностного импеданса с
  сопротивлением лёгкого и изменение вкладов других органов.
  Это обосновывает необходимость прямой FEM-проверки, но не
  выбирает модель смешения и её параметры.
- Dipa et al., *Effects of temperature on electrical impedance of
  biological tissues: ex-vivo measurements*
  (Zotero SMJAE552, DOI 10.2478/joeb-2024-0013) показывают
  зависимость импеданса биологических образцов от частоты и
  температуры. Работа не даёт свойства лёгочной матрицы, но
  подтверждает, что эти условия нельзя опускать.

Прямой первичный источник, который валидирует
Максвелла–Гарнетта или адаптацию Арчи для лёгкого при частоте
реокардиомониторов и задаёт свойства неаэрированной матрицы, в
текущем локальном индексе не найден. Число
$\rho_{\mathrm{matrix}}=2.5$ Ом·м и показатель $m=1.5$ остаются
историческими допущениями, а не литературно подтверждёнными
входами.


## Проверки и критерий завершения

Исполняемая реализация должна пройти:

1. тест предела $f=0$, где модель возвращает свойство матрицы;
2. проверку положительности, монотонности и поведения при
   $f\to1$ без молчаливого обрезания;
3. сверку частного скалярного выражения Максвелла–Гарнетта с
   общей комплексной формулой;
4. анализ чувствительности к $f$, частоте, свойствам матрицы и
   параметрам модели;
5. сравнение с прямым FEM на зарегистрированной КТ-геометрии;
6. только затем — передача сценариев в
   34.03_КТ_как_референс_разработки.md для слепого сопоставления
   с электрической обратной задачей без настройки параметров по
   этому же результату.

Совпадение с одной импедансной оценкой считается перекрёстной
согласованностью, но не подтверждением всей двуслойной модели.
Этап может получить статус accepted только после появления
принятого 20.03, установленной частоты, обоснованных свойств
матрицы, выполнения внутренних тестов и прямой FEM-проверки.
Статус межмодальной согласованности присваивается отдельно в 34.03.
